# Minimal temporal modeling comparison

This notebook compares four small PyTorch models on two synthetic sequence tasks:

- `MLP`: timestep-independent `Linear -> ReLU -> Linear`
- `MLP + DelayLaggedMixer`: your current fixed one-lag-per-connection delayed mixer followed by the same MLP
- `GRU`: a standard 1-layer GRU with a linear readout
- `TCN`: a small causal dilated convolution stack

The two tasks share the same train/validation sequence split:

- Temporal: a noisier nonlinear function of several past timesteps
- Instantaneous: a noisier nonlinear function of only the current timestep


In [1]:
import math

import torch
import torch.nn as nn


class DelayLaggedMixer(nn.Module):
    def __init__(
        self,
        channels,
        lags=(0, 1, 2, 3, 4, 5, 6, 7),
        one_random_lag_per_pair=True,
        num_lags_per_connection=1,
        lag_distribution_type="uniform",
        lag_decay_strength=1.0,
        seed=0,
        bias=True,
    ):
        super().__init__()

        if not one_random_lag_per_pair or num_lags_per_connection != 1:
            raise NotImplementedError(
                "DelayLaggedMixer supports exactly one fixed lag per connection."
            )

        self.channels = int(channels)
        self.lags = [int(lag) for lag in lags]
        self.K = len(self.lags)
        self.lag_distribution_type = str(lag_distribution_type)
        self.lag_decay_strength = float(lag_decay_strength)

        lag_probs = self._normalized_lag_probabilities()

        with torch.random.fork_rng():
            torch.manual_seed(int(seed))
            lag_indices = torch.multinomial(
                lag_probs.expand(self.channels * self.channels, -1),
                num_samples=1,
                replacement=False,
            ).view(self.channels, self.channels)

        self.register_buffer("lag_indices", lag_indices)
        self.register_buffer(
            "lag_values",
            torch.tensor(self.lags, dtype=torch.long)[lag_indices],
        )

        self.weight = nn.Parameter(torch.empty(self.channels, self.channels))
        self.bias = nn.Parameter(torch.zeros(self.channels)) if bias else None

        masks = []
        for k in range(self.K):
            masks.append((self.lag_indices == k).float())
        self.register_buffer("lag_masks", torch.stack(masks))

        self.reset_parameters()

    def _normalized_lag_probabilities(self):
        lag_values = torch.tensor(self.lags, dtype=torch.float32)

        if self.lag_distribution_type == "uniform":
            probs = torch.ones_like(lag_values)
        elif self.lag_distribution_type == "short-biased":
            probs = lag_values.max() - lag_values + 1.0
        elif self.lag_distribution_type == "long-biased":
            probs = lag_values - lag_values.min() + 1.0
        elif self.lag_distribution_type == "center-biased":
            center = lag_values.mean()
            probs = 1.0 / (1.0 + (lag_values - center).abs())
        elif self.lag_distribution_type == "decaying":
            probs = torch.exp(-self.lag_decay_strength * (lag_values - lag_values.min()))
        else:
            raise ValueError(f"Unsupported lag_distribution_type: {self.lag_distribution_type}")

        return probs / probs.sum()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            bound = 1.0 / math.sqrt(self.channels)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        T, B, C = x.shape
        if C != self.channels:
            raise ValueError(f"Expected channels={self.channels}, got {C}")

        y = x.new_zeros(T, B, C)

        for k, lag in enumerate(self.lags):
            W_k = self.weight * self.lag_masks[k].to(self.weight.dtype)

            if lag == 0:
                x_shift = x
            else:
                x_shift = x.new_zeros(T, B, C)
                x_shift[lag:] = x[:-lag]

            y = y + torch.matmul(x_shift, W_k.T)

        if self.bias is not None:
            y = y + self.bias.view(1, 1, -1)

        return y


In [2]:
from pathlib import Path
import random
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training" / "models" / "mlp_backbone.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root(Path.cwd())
MODEL_TRAINING_DIR = ROOT / "model_training"
if str(MODEL_TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_TRAINING_DIR))


SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_DIM = 16
SEQ_LEN = 64
TRAIN_SAMPLES = 1024
VAL_SAMPLES = 256
BATCH_SIZE = 32
EPOCHS = 12
LR = 3e-3
TASK_NOISE_STD = 0.35

print(f"Repo root: {ROOT}")
print(f"Using device: {DEVICE}")
print(
    "Model widths chosen for rough parameter parity: "
    "MLP hidden=128, DelayLaggedMixer MLP hidden=128, GRU hidden=20, TCN channels=16"
)


Repo root: /home/alex/snn_for_bts/brain_to_text
Using device: cuda
Model widths chosen for rough parameter parity: MLP hidden=128, DelayLaggedMixer MLP hidden=128, GRU hidden=20, TCN channels=16


In [3]:
def shift_sequence(x: torch.Tensor, lag: int) -> torch.Tensor:
    if lag == 0:
        return x
    pad = torch.zeros(x.size(0), lag, x.size(2), dtype=x.dtype, device=x.device)
    return torch.cat([pad, x[:, :-lag, :]], dim=1)


def make_datasets():
    x_train = torch.randn(TRAIN_SAMPLES, SEQ_LEN, INPUT_DIM)
    x_val = torch.randn(VAL_SAMPLES, SEQ_LEN, INPUT_DIM)

    def make_targets(x: torch.Tensor):
        lag1_feature4 = shift_sequence(x, 1)[..., 4]
        lag2_feature0 = shift_sequence(x, 2)[..., 0]
        lag3_feature2 = shift_sequence(x, 3)[..., 2]
        lag5_feature1 = shift_sequence(x, 5)[..., 1]
        lag7_feature3 = shift_sequence(x, 7)[..., 3]

        temporal_score = (
            0.9 * lag2_feature0
            - 0.8 * lag5_feature1
            + 0.6 * lag3_feature2 * lag7_feature3
            + 0.4 * lag1_feature4
            + TASK_NOISE_STD * torch.randn_like(lag2_feature0)
        )

        instantaneous_score = (
            0.8 * x[..., 0] * x[..., 1]
            + 0.9 * x[..., 2]
            - 0.7 * x[..., 3]
            + 0.4 * torch.tanh(x[..., 4])
            + TASK_NOISE_STD * torch.randn_like(x[..., 0])
        )

        temporal = (temporal_score > 0).float().unsqueeze(-1)
        instantaneous = (instantaneous_score > 0).float().unsqueeze(-1)
        return temporal, instantaneous

    y_train_temporal, y_train_instantaneous = make_targets(x_train)
    y_val_temporal, y_val_instantaneous = make_targets(x_val)

    return {
        "temporal": (x_train, y_train_temporal, x_val, y_val_temporal),
        "instantaneous": (x_train, y_train_instantaneous, x_val, y_val_instantaneous),
    }


class TimewiseMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DelayLaggedMixerMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, seed: int):
        super().__init__()
        self.mixer = DelayLaggedMixer(
            channels=input_dim,
            lags=[0, 1, 2, 3, 4, 5, 6, 7, 8],
            one_random_lag_per_pair=True,
            num_lags_per_connection=1,
            lag_distribution_type="uniform",
            lag_decay_strength=1.0,
            seed=seed,
        )
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.mixer(x.transpose(0, 1).contiguous()).transpose(0, 1).contiguous()
        return self.net(x)


class GRUModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.readout = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden, _ = self.gru(x)
        return self.readout(hidden)


class CausalConv1d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, dilation: int):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=self.padding,
            dilation=dilation,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        if self.padding > 0:
            x = x[..., :-self.padding]
        return x


class TCNModel(nn.Module):
    def __init__(self, input_dim: int, hidden_channels: int, kernel_size: int = 3):
        super().__init__()
        dilations = [1, 2, 4]
        layers = []
        in_channels = input_dim
        for dilation in dilations:
            layers.append(CausalConv1d(in_channels, hidden_channels, kernel_size, dilation))
            layers.append(nn.ReLU())
            in_channels = hidden_channels
        self.tcn = nn.Sequential(*layers)
        self.readout = nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2).contiguous()
        x = self.tcn(x)
        x = x.transpose(1, 2).contiguous()
        return self.readout(x)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def make_train_loader(x: torch.Tensor, y: torch.Tensor) -> DataLoader:
    return DataLoader(
        TensorDataset(x, y),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        generator=torch.Generator().manual_seed(SEED),
    )


def make_val_loader(x: torch.Tensor, y: torch.Tensor) -> DataLoader:
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=False)


def evaluate_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            predictions = (logits > 0).float()
            correct += (predictions == yb).sum().item()
            total += yb.numel()
    return correct / total


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader) -> float:
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

    return evaluate_accuracy(model, val_loader)


MODEL_FACTORIES = {
    "MLP": lambda: TimewiseMLP(INPUT_DIM, hidden_dim=128),
    "MLP + DelayLaggedMixer": lambda: DelayLaggedMixerMLP(
        INPUT_DIM,
        hidden_dim=128,
        seed=SEED,
    ),
    "GRU": lambda: GRUModel(INPUT_DIM, hidden_dim=20, num_layers=1),
    "TCN": lambda: TCNModel(INPUT_DIM, hidden_channels=16, kernel_size=3),
}


def run_experiment():
    datasets = make_datasets()
    results = {
        name: {
            "params": count_parameters(factory()),
            "temporal": None,
            "instantaneous": None,
        }
        for name, factory in MODEL_FACTORIES.items()
    }

    for task_name, (x_train, y_train, x_val, y_val) in datasets.items():
        for name, factory in MODEL_FACTORIES.items():
            train_loader = make_train_loader(x_train, y_train)
            val_loader = make_val_loader(x_val, y_val)
            torch.manual_seed(SEED)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(SEED)
            model = factory()
            accuracy = train_model(model, train_loader, val_loader)
            results[name][task_name] = accuracy

    return results


def print_results_table(results):
    header = f"{'Model':<28} | {'Params':>6} | {'Temporal Task Accuracy':>24} | {'Instantaneous Task Accuracy':>29}"
    divider = "-" * len(header)
    print(header)
    print(divider)
    for model_name, metrics in results.items():
        print(
            f"{model_name:<28} | "
            f"{metrics['params']:>6d} | "
            f"{metrics['temporal']:>24.4f} | "
            f"{metrics['instantaneous']:>29.4f}"
        )


results = run_experiment()
print_results_table(results)


Model                        | Params |   Temporal Task Accuracy |   Instantaneous Task Accuracy
------------------------------------------------------------------------------------------------
MLP                          |   2305 |                   0.4967 |                        0.9128
MLP + DelayLaggedMixer       |   2577 |                   0.7446 |                        0.8948
GRU                          |   2301 |                   0.8432 |                        0.9130
TCN                          |   2369 |                   0.8471 |                        0.9086
